# 실험 C - CoAtNet-2

기준: 실험 5 (DINOv2-S 224px) val 0.8420

캐시, 레이블, epoch 은 같다. 백본과 lr 만 다르다


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

T0 = time.time()
for _c in ['/kaggle/input/datasets/hyejinkwak/knee-src', '/kaggle/input/knee-src']:
    if os.path.isdir(_c):
        sys.path.insert(0, _c); break

import torch, numpy as np, pandas as pd
import paths, config, dicom_io, dataset as ds, labels as lb, model as M, train as T

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    cc = torch.cuda.get_device_capability(0)
    print('compute capability', cc)
    assert cc >= (7, 0), ('P100(sm_60)은 사전설치 PyTorch 와 호환되지 않는다. '
                          'kernel-metadata.json 에 machine_shape=NvidiaTeslaT4 를 넣을 것')
config.verify_labels()
print(paths.describe())


## 설정

`DEBUG=True` 면 스터디 64건, 1 epoch 만 돈다


In [ ]:
DEBUG = False     # 본 학습. True 면 스터디 64건 1 epoch 스모크
RUN_BENCH = False # 속도 측정 (이미 측정함: 읽기 230 vs 연산 14.7 study/s, 병목은 연산)

# 루트 노트북(pilkwang, 0.891)과 대조해 맞춘 것들
#   LR_BACKBONE 8e-6  "the encoder is adapted, not retrained"
#   UNFREEZE_LAST 6, WEIGHT_DECAY 0.02, EPOCHS 10
#   SlotHead (MultiheadAttention 대신 einsum attention, 0.7M -> 0.107M)
#   침묵 셀 가중 0.3 -> 0.05, 손실은 (BCE*w).mean()
#   9슬라이스 중 한 그룹만 학습 (인코더 입력 18장 -> 6장, 3배 빠름)
#   OneCycleLR pct_start 0.15
#   증강 회전 ±8도 / 스케일 ±8% / 이동 ±5% / 밝기 ±10%
#   SLOT_PRIOR_TABLE 고정 buffer
# 10 epoch 에서 val 0.8260 / gold 0.8487 로 끝났는데 아직 수렴하지 않았다
#   ep7 0.8234 -> ep8 0.8260 -> ep9 0.8257 (val 은 평평)
#   gold 는 0.8445 -> 0.8437 -> 0.8487 로 계속 오름
#   loss 도 0.4086 -> 0.4010 으로 계속 내려감
# 83초/epoch 이라 30 epoch 이 42분이다
EPOCHS = 30

CACHE = paths.find_input('knee-cache-build')
print('cache:', CACHE)

# 실험 C - CoAtNet-2 백본 (journey.md 9장)
#   기준은 실험 5 (DINOv2-S, 같은 224 캐시) val 0.8420
#   목적 두 가지 - 단독 성능, 그리고 DINOv2 와 다르게 틀리는 앙상블 멤버
#   공개 앙상블 핵심 멤버와 같은 계열 (공개는 384px, 우리 캐시는 224px)
#   CoAtNet 단일 fold 0.950, 0.949 보고 (디스커션 735304)
#   73M 으로 ResNet50 의 3배라 encoder lr 5e-5, head 1e-3, batch 4 x accum 4
cfg = T.TrainConfig(
    fold=2, plan='C', epochs=EPOCHS, batch_size=4, accum=4,
    lr=5e-5, head_lr_mult=20.0,
    num_workers=4, debug=DEBUG, out_dir='/kaggle/working')

wd = paths.find_input('knee-cnn-weights')
WEIGHTS = str(Path(wd) / 'coatnet_rmlp_2_rw_224_224.pth')
assert Path(WEIGHTS).exists(), f'{WEIGHTS} 없음'
wmeta = json.loads((Path(wd) / 'weights_meta_coatnet_rmlp_2_rw_224.json').read_text())
assert wmeta['model'] == 'coatnet_rmlp_2_rw_224.sw_in12k_ft_in1k', wmeta['model']
print('offline weights:', WEIGHTS, wmeta['model'])

mcfg = M.ModelConfig(backbone='coatnet_rmlp_2_rw_224.sw_in12k_ft_in1k', pretrained=False,
                     weights_path=WEIGHTS)
print(cfg)
print(mcfg)


## 캐시 확인

슬롯 수가 코드와 다르면 즉시 멈춘다


In [ ]:
cache = ds.open_cache(CACHE)
print('shards', len(cache.arrays), '| studies', len(cache.index))
print('slots', cache.slots)

pick, mask = ds.build_slot_map(cache, cfg.plan)
cov = pd.DataFrame(mask, columns=[f'slot{i}' for i in range(mask.shape[1])])
print(f'\nplan {cfg.plan} 칸별 채움률')
print(cov.mean().round(3).to_string())
print(f'평균 {cov.mean().mean():.3f}')


## 속도 측정

본 학습 시간을 추정하려면 **읽기와 연산을 따로 재야** 한다

마운트 검증에서 랜덤 접근이 78.4 ms/study 로 나왔다 (1 epoch 읽기만 5.8분)
ShardSequentialSampler 와 num_workers 가 이걸 얼마나 줄이는지 확인한다

**이 셀 결과로 num_workers 와 epochs 를 정한다**


In [ ]:
if RUN_BENCH:
    from torch.utils.data import DataLoader

    N_BENCH = 256
    y_b, w_b = lb.load_targets()
    bench_studies = [s for s in cache.index.study if s in y_b.index][:N_BENCH]
    bench_ds = ds.KneeCacheDataset(cache, bench_studies, y_b, w_b,
                                   plan=cfg.plan, augment=True)

    def time_loader(nw, sequential=True, bs=8):
        sampler = ds.ShardSequentialSampler(bench_ds, shuffle=True) if sequential else None
        dl = DataLoader(bench_ds, batch_size=bs, sampler=sampler,
                        shuffle=None if sequential else True,
                        num_workers=nw, pin_memory=True,
                        persistent_workers=nw > 0)
        next(iter(dl))                      # 워커 기동 비용 제외
        t = time.time(); n = 0
        for b in dl:
            n += b['image'].shape[0]
        el = time.time() - t
        return el, n / el, el / n * 1000

    print('데이터 읽기만 (모델 없음)')
    read_rates = {}
    for nw in [0, 2, 4]:
        el, rate, ms = time_loader(nw)
        read_rates[nw] = rate
        print(f'  num_workers={nw}  {rate:6.1f} study/s  {ms:5.1f} ms/study  '
              f'-> 1 epoch(3525건) {3525/rate/60:.1f}분')
else:
    print('속도 측정 건너뜀 (RUN_BENCH=False)')


In [ ]:
if RUN_BENCH:
    print('샤드 순차 vs 완전 무작위 (num_workers=4)')
    for seq in [True, False]:
        el, rate, ms = time_loader(4, sequential=seq)
        print(f'  {"샤드 순차" if seq else "완전 무작위"}  {rate:6.1f} study/s  {ms:5.1f} ms/study')
else:
    print('속도 측정 건너뜀 (RUN_BENCH=False)')


In [ ]:
if RUN_BENCH:
    # 학습 step 시간 (읽기 제외)
    net_b = M.KneeModel(M.ModelConfig(**{**mcfg.__dict__,
                                         'n_slots': len(ds.resolve_plan(cfg.plan))})).cuda()
    opt_b = torch.optim.AdamW(net_b.parameters(), lr=1e-4)
    scaler_b = torch.amp.GradScaler('cuda')

    img = torch.randint(0, 255, (cfg.batch_size, len(ds.resolve_plan(cfg.plan)), 9, 224, 224),
                        dtype=torch.uint8).cuda()
    msk = torch.ones(cfg.batch_size, len(ds.resolve_plan(cfg.plan))).cuda()
    yb = torch.rand(cfg.batch_size, 12).cuda(); wb = torch.ones_like(yb)

    for _ in range(3):                      # 워밍업
        with torch.autocast('cuda'):
            loss = M.masked_bce(net_b(img, msk), yb, wb)
        scaler_b.scale(loss).backward(); scaler_b.step(opt_b); scaler_b.update()
        opt_b.zero_grad(set_to_none=True)
    torch.cuda.synchronize()

    t = time.time(); STEPS = 10
    for _ in range(STEPS):
        with torch.autocast('cuda'):
            loss = M.masked_bce(net_b(img, msk), yb, wb)
        scaler_b.scale(loss).backward(); scaler_b.step(opt_b); scaler_b.update()
        opt_b.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    step_s = (time.time() - t) / STEPS
    compute_rate = cfg.batch_size / step_s

    print(f'학습 step {step_s*1000:.0f} ms  ({compute_rate:.1f} study/s)')
    print(f'GPU 메모리 {torch.cuda.max_memory_allocated()/1024**3:.2f} GiB')
    del net_b, opt_b, img, msk; torch.cuda.empty_cache()


In [ ]:
if RUN_BENCH:
    # 전체 학습 시간 추정
    best_nw = max(read_rates, key=read_rates.get)
    read_rate = read_rates[best_nw]
    eff = min(read_rate, compute_rate)        # 병렬이면 느린 쪽이 병목
    bound = '읽기' if read_rate < compute_rate else '연산'

    n_train, n_valid = 3525, 882
    per_epoch = n_train/eff + n_valid/compute_rate/2
    print(f'병목: {bound}  (읽기 {read_rate:.1f} vs 연산 {compute_rate:.1f} study/s)')
    print(f'권장 num_workers = {best_nw}')
    print(f'\nepoch 당 {per_epoch/60:.1f}분')
    for e in [4, 6, 8, 12]:
        print(f'  {e:>2} epoch -> {per_epoch*e/60:.0f}분  (GPU {per_epoch*e/3600:.2f}h)')

    json.dump({'read_study_per_s': {str(k): round(v,1) for k,v in read_rates.items()},
               'compute_study_per_s': round(compute_rate,1),
               'step_ms': round(step_s*1000),
               'bottleneck': bound, 'best_num_workers': best_nw,
               'minutes_per_epoch': round(per_epoch/60, 2)},
              open('/kaggle/working/speed_report.json','w'), indent=2)
    print('\nsaved speed_report.json')


## 학습


In [ ]:
report = T.run(str(CACHE), cfg, mcfg)
print()
print(json.dumps(report['best'], indent=2))


## 결과


In [ ]:
h = pd.DataFrame([{k: v for k, v in r.items() if k != 'per_label'}
                  for r in report['history']])
print(h.to_string(index=False))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(h.epoch, h.loss, marker='o'); ax[0].set_title('train loss', fontsize=9)
ax[1].plot(h.epoch, h.val_auc, marker='o', label='valid (report labels)')
if h.gold_auc.notna().any():
    ax[1].plot(h.epoch, h.gold_auc, marker='s', label='gold panel')
ax[1].set_title('macro AUC', fontsize=9); ax[1].legend(fontsize=8)
for a in ax: a.set_xlabel('epoch')
plt.tight_layout()
plt.savefig('/kaggle/working/train_curve.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
best = max(report['history'], key=lambda r: r['val_auc'])
per = pd.Series(best['per_label']).sort_values()
print('레이블별 AUC (best epoch)')
print(per.round(4).to_string())
print(f'\nmacro {best["val_auc"]:.4f}')
print()
print('검증 타깃이 판독문 유래라는 점을 기억할 것')
print('이 수치는 판독문을 얼마나 잘 맞추나이지 무릎을 얼마나 잘 보나가 아니다')
print(f'\n총 소요 {(time.time()-T0)/60:.1f}분')
print('output:', sorted(os.listdir('/kaggle/working')))
